# SmartMovie — Étape 4 : Feature Engineering

Ce notebook repart des données **MongoDB** (source de vérité), réapplique le nettoyage de base fait en EDA, puis construit de nouvelles variables utiles pour la suite (modélisation / recommandation).

Chaque variable est justifiée. Une section dédiée à la fin traite explicitement du **data leakage** : certaines colonnes du dataset ne sont connues qu'*après* la sortie du film (notes, votes, popularité, revenu) et ne doivent jamais servir de features si on cherche à prédire un résultat lié au succès du film — sinon le modèle "triche" en utilisant une information qu'il n'aurait pas au moment de la prédiction réelle.


## 1. Chargement et nettoyage de base (identique à l'EDA)

In [22]:
import os
import numpy as np
import pandas as pd
from pymongo import MongoClient
from dotenv import load_dotenv

load_dotenv()

MONGO_URI = os.getenv("MONGO_URI")
client = MongoClient(MONGO_URI)
db = client["smartmovie"]
collection = db["movies"]

df = pd.DataFrame(list(collection.find({}, {"_id": 0})))
print(f"{len(df)} films chargés")


931 films chargés


In [23]:
# Nettoyage de base (identique au notebook EDA)
df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")

for col in ["budget", "revenue", "runtime"]:
    if col in df.columns:
        df[col] = df[col].replace(0, np.nan)

df["genres_list"] = df["genres"].fillna("").apply(
    lambda x: [g.strip() for g in x.split(",") if g.strip()]
)
df["keywords_list"] = df["keywords"].fillna("").apply(
    lambda x: [k.strip() for k in x.split(",") if k.strip()]
)

df.head()


,movie_id,budget,genres,keywords,original_language,overview,popularity,release_date,revenue,runtime,title,vote_average,vote_count,genres_list,keywords_list
0,969681,225000000.0,"Science Fiction, Action, Adventure","mind control, new york city, hero, mutation, s...",en,Fighting crime full-time as Spider-Man in a wo...,687.7014,1970-01-01 00:29:45.283200,2.495683e+09,145.0,Spider-Man: Brand New Day,7.865,2923,"[Science Fiction, Action, Adventure]","[mind control, new york city, hero, mutation, ..."
1,1423191,75000000.0,"Horror, Science Fiction, Adventure","outbreak, survival, zombie, based on video gam...",en,Medical courier Bryan unwittingly finds himsel...,407.7241,1970-01-01 00:29:49.516800,1.964762e+08,95.0,Resident Evil,7.300,556,"[Horror, Science Fiction, Adventure]","[outbreak, survival, zombie, based on video ga..."
2,1368337,250000000.0,"Adventure, Action, Fantasy","ship, trojan war, greek mythology, historical ...",en,"Odysseus, the legendary King of Ithaca, embark...",366.2926,1970-01-01 00:29:44.073600,1.751248e+09,173.0,The Odyssey,8.019,3946,"[Adventure, Action, Fantasy]","[ship, trojan war, greek mythology, historical..."
3,1751701,100000.0,"Science Fiction, Drama, Thriller","asteroid, loss, love, loneliness, disaster, de...",en,"After losing his father, Ethan Nichols inherit...",265.7821,1970-01-01 00:29:49.430400,NaN,95.0,After Impact,6.281,16,"[Science Fiction, Drama, Thriller]","[asteroid, loss, love, loneliness, disaster, d..."
4,1084244,250000000.0,"Animation, Family, Comedy, Adventure","farm, island, pig, horse, rescue mission, riva...",en,When Bonnie receives a Lilypad tablet as a gif...,198.3883,1970-01-01 00:29:41.654400,1.136476e+09,102.0,Toy Story 5,8.319,2389,"[Animation, Family, Comedy, Adventure]","[farm, island, pig, horse, rescue mission, riv..."


## 2. Variables temporelles : année, mois, décennie

**Pourquoi ces variables ?**
- `release_year` / `release_month` / `release_decade` permettent de capter des effets saisonniers et des tendances de long terme (évolution du genre, inflation des budgets, etc.) sans dépendre de la colonne `release_date` brute que la plupart des modèles ne savent pas exploiter directement.
- Le **mois** est pertinent car le marché du cinéma est saisonnier (sorties "blockbuster" en été / fêtes de fin d'année aux US) — il peut expliquer une partie de la variance de `revenue` ou `popularity` indépendamment de la qualité du film.
- La **décennie** permet de regrouper les films par époque sans avoir une variable trop fine (`release_year` seul) qui génère trop de catégories avec peu d'exemples chacune.

**Pas de leakage ici** : la date de sortie est connue *avant* la sortie du film (c'est une donnée de planning), donc ces variables sont utilisables sans risque, quel que soit l'objectif de prédiction.


In [24]:
df["release_year"] = df["release_date"].dt.year
df["release_month"] = df["release_date"].dt.month
df["release_decade"] = (df["release_year"] // 10 * 10)

# Saisonnalité : période "été" (juin-août) et "fêtes de fin d'année" (nov-déc),
# connues pour concentrer les plus gros succès au box-office
df["is_summer_release"] = df["release_month"].isin([6, 7, 8]).astype(int)
df["is_holiday_release"] = df["release_month"].isin([11, 12]).astype(int)

df[["release_date", "release_year", "release_month", "release_decade",
    "is_summer_release", "is_holiday_release"]].head()


,release_date,release_year,release_month,release_decade,is_summer_release,is_holiday_release
0,1970-01-01 00:29:45.283200,1970,1,1970,0,0
1,1970-01-01 00:29:49.516800,1970,1,1970,0,0
2,1970-01-01 00:29:44.073600,1970,1,1970,0,0
3,1970-01-01 00:29:49.430400,1970,1,1970,0,0
4,1970-01-01 00:29:41.654400,1970,1,1970,0,0


## 3. Nombre de genres et de mots-clés

**Pourquoi ces variables ?**
- `n_genres` : un film multi-genres (ex. "Action, Comédie, Aventure") cible potentiellement un public plus large qu'un film mono-genre. C'est un indicateur simple de positionnement marketing / diversité de contenu.
- `n_keywords` : TMDB associe plus de mots-clés aux films qui ont une fiche bien renseignée / qui ont une identité narrative riche (univers, thèmes récurrents). Peut aussi être un proxy (imparfait) de la popularité de la fiche TMDB elle-même.

**Pas de leakage** : ce sont des métadonnées descriptives du film (genres, mots-clés thématiques), connues dès l'annonce du film, pas des mesures de performance.


In [25]:
df["n_genres"] = df["genres_list"].apply(len)
df["n_keywords"] = df["keywords_list"].apply(len)
df["has_keywords"] = (df["n_keywords"] > 0).astype(int)

df[["title", "n_genres", "n_keywords", "has_keywords"]].head()


,title,n_genres,n_keywords,has_keywords
0,Spider-Man: Brand New Day,3,19,1
1,Resident Evil,3,11,1
2,The Odyssey,3,19,1
3,After Impact,3,7,1
4,Toy Story 5,4,27,1


## 4. Catégories de durée (`runtime`)

**Pourquoi cette variable ?**
Transformer `runtime` (continu) en catégories permet de capter des effets non-linéaires simples (ex. un film "très long" n'a pas un effet proportionnel à sa durée, mais un effet de seuil — fatigue du spectateur, moins de séances par jour en salle, etc.) et rend la variable plus interprétable dans une analyse ou un modèle simple (arbre, régression logistique).

Seuils choisis (standards dans l'industrie) :
- < 90 min : Court
- 90–120 min : Standard
- 120–150 min : Long
- > 150 min : Très long

**Pas de leakage** : la durée finale du film est fixée au montage, bien avant la sortie — c'est une caractéristique du produit, pas un résultat de son succès.


In [26]:
bins = [0, 90, 120, 150, np.inf]
labels = ["Court (<90min)", "Standard (90-120min)", "Long (120-150min)", "Très long (>150min)"]

df["runtime_category"] = pd.cut(df["runtime"], bins=bins, labels=labels)

df["runtime_category"].value_counts()


runtime_category
Standard (90-120min)    469
Long (120-150min)       276
Court (<90min)           94
Très long (>150min)      84
Name: count, dtype: int64

## 5. Autres indicateurs pertinents

- **`overview_length`** : nombre de mots du synopsis. Un synopsis plus détaillé peut refléter un film avec une fiche mieux travaillée côté marketing/studio — connu avant la sortie (le synopsis est écrit pour la promotion).
- **`is_original_language_en`** : indicateur binaire "film en anglais ou non". Pertinent car le marché anglophone domine la distribution internationale sur TMDB ; peut capter un biais structurel d'accès au marché plutôt qu'une caractéristique de qualité.
- **`budget_category`** : catégorisation du budget en quartiles (`qcut`, pas des seuils fixes, car `budget` est très asymétrique — voir EDA). Plus interprétable que la valeur brute, et robuste aux valeurs extrêmes (quelques budgets énormes n'écrasent pas la variable).

Ces trois variables sont connues **avant** la sortie du film (synopsis écrit en amont, langue fixée au tournage, budget décidé en pré-production) : pas de leakage.


In [27]:
df["overview_length"] = df["overview"].fillna("").apply(lambda x: len(x.split()))

df["is_original_language_en"] = (df["original_language"] == "en").astype(int)


df[["overview_length", "is_original_language_en"]].head()


,overview_length,is_original_language_en
0,68,1
1,19,1
2,46,1
3,91,1
4,39,1


## 6. ⚠️ Data Leakage : quelles colonnes NE PAS utiliser comme features

Le **data leakage** se produit quand une variable utilisée pour entraîner le modèle contient, directement ou indirectement, une information qui ne serait pas disponible au moment réel de la prédiction — ou qui est elle-même une conséquence de la cible qu'on cherche à prédire. Le modèle obtient alors des performances artificiellement excellentes en entraînement/test, mais s'effondre en production car cette information n'existe pas encore au moment de la prédiction.

Dans ce dataset, plusieurs colonnes sont calculées **après** la sortie du film et sont donc dangereuses selon ce qu'on cherche à prédire :

| Colonne | Quand est-elle connue ? | Risque |
|---|---|---|
| `vote_average`, `vote_count` | Après la sortie, s'accumulent avec le temps | Si la cible est `popularity` ou `revenue` : ces votes sont la **conséquence** du succès du film, pas une cause. Les utiliser comme feature revient à utiliser le résultat pour prédire le résultat. |
| `popularity` | Mesurée en continu par TMDB, reflète l'engouement *actuel* | Ne jamais l'utiliser comme feature si c'est elle (ou une variable très corrélée comme `revenue`) qui est la cible. |
| `revenue` | Connu seulement après l'exploitation en salle | Si la cible est `revenue` lui-même, c'est trivialement la cible — jamais en feature. Si la cible est `popularity`, `revenue` reste un leakage indirect fort (très corrélés, voir heatmap de l'EDA). |

**Règle pratique appliquée ici :**
- Pour un cas d'usage de type *"prédire le succès d'un film avant sa sortie"* (`revenue` et/ou `popularity` comme cible), on construit un jeu de features **strictement pré-sortie** : métadonnées, budget, durée, genres, mots-clés, langue, saisonnalité. On exclut `vote_average`, `vote_count`, `popularity` et `revenue` des features (ces deux dernières ne sont utilisées que comme cibles possibles, jamais les deux en même temps vu leur forte corrélation).
- Pour un cas d'usage de type *recommandation de contenu* (similarité entre films), ce risque ne s'applique pas de la même façon puisqu'on ne prédit pas un résultat futur — mais la distinction reste utile à documenter.

On matérialise ci-dessous un jeu de features "sûres" (pré-sortie), séparé des colonnes "post-sortie" à ne jamais mélanger dedans.


In [28]:
pre_release_features = [
    "movie_id", "title",
    "release_year", "release_month", "release_decade",
    "is_summer_release", "is_holiday_release",
    "n_genres", "n_keywords", "has_keywords",
    "runtime", "runtime_category",
    "overview_length", "is_original_language_en",
    "budget", 
]

post_release_targets = ["vote_average", "vote_count", "popularity", "revenue"]

df_features = df[pre_release_features].copy()
df_targets = df[post_release_targets].copy()

print("Features pré-sortie (utilisables en entrée du modèle) :")
print(df_features.columns.tolist())
print("\nCibles / colonnes post-sortie (jamais en feature en même temps que leur propre cible) :")
print(df_targets.columns.tolist())

df_features.info()


Features pré-sortie (utilisables en entrée du modèle) :
['movie_id', 'title', 'release_year', 'release_month', 'release_decade', 'is_summer_release', 'is_holiday_release', 'n_genres', 'n_keywords', 'has_keywords', 'runtime', 'runtime_category', 'overview_length', 'is_original_language_en', 'budget']

Cibles / colonnes post-sortie (jamais en feature en même temps que leur propre cible) :
['vote_average', 'vote_count', 'popularity', 'revenue']
<class 'pandas.DataFrame'>
RangeIndex: 931 entries, 0 to 930
Data columns (total 15 columns):
 #   Column                   Non-Null Count  Dtype   
---  ------                   --------------  -----   
 0   movie_id                 931 non-null    int64   
 1   title                    931 non-null    str     
 2   release_year             931 non-null    int32   
 3   release_month            931 non-null    int32   
 4   release_decade           931 non-null    int32   
 5   is_summer_release        931 non-null    int64   
 6   is_holiday_rele

## 7. Sauvegarde

On enregistre le jeu de features (et les cibles, séparément) dans une nouvelle collection MongoDB, pour garder le pipeline cohérent avec les étapes précédentes et faciliter la suite (modélisation).


In [29]:
from pymongo import UpdateOne

df_export = df_features.merge(df_targets, left_index=True, right_index=True)
# NaN -> None pour un stockage Mongo propre
df_export = df_export.where(pd.notnull(df_export), None)

features_collection = db["movies_features"]

operations = [
    UpdateOne({"movie_id": row["movie_id"]}, {"$set": row}, upsert=True)
    for row in df_export.to_dict("records")
]

if operations:
    features_collection.bulk_write(operations)
    features_collection.create_index("movie_id", unique=True)

print(f"{len(operations)} documents upsertés dans 'movies_features'")


931 documents upsertés dans 'movies_features'
